# CV-DV ECD-VQE: how much depth and how many iterations?

**Status: preliminary.** Cells here are 8 random starts each unless noted; the
50-seed runs are queued. Read the *directions* as findings and the *magnitudes*
as provisional -- 2/8 has a 95% interval of roughly 3-65%.

The DV comparison arm is deliberately absent. `scaling_dv_adam_n12.npz` was
never written (the job was OOM-killed), n=10 DV has never run, and the n=8 DV
baseline sits at a different iteration budget, so any comparison drawn today
would be answering a question we have not measured.

### What prompted this

`run_scaling_sweeps` pins `MULTISTART_ITERS = 8000` at every problem size and
every depth. Nobody had checked whether that is enough. It is at 8 variables,
and it is not at 12 -- which means the published CV-DV success rates at
n >= 12 describe an under-trained ansatz rather than the ansatz.

Two words that mean different things here:

- **`budget_used`** -- fraction of the *iteration* budget a run consumed before
  it stopped improving. High means the optimizer was still descending when the
  budget ended.
- **truncation / `tail_population`** -- oscillator population in the top Fock
  levels, i.e. whether the *cutoff* is wide enough. Measured at 2x cutoff by
  `--validate`; it comes out around 1e-34, so the cutoff is not a concern.

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

DATA = Path("data/iteration_study")
if not DATA.exists():                      # notebook run from the repo root
    DATA = Path("cvdv_vs_dv/data/iteration_study")

SUCCESS = 0.9        # P(optimal) counting as solved; matches KNEE_THRESHOLD
SIZES = (8, 10, 12)

# Validated categorical slots 1-3 (all-pairs CVD dE 9.2, normal-vision 24.0).
SERIES = {8: "#2a78d6", 10: "#eb6834", 12: "#1baf7a"}
SEQ = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
SURFACE, INK, INK2 = "#fcfcfb", "#0b0b0b", "#52514e"

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "text.color": INK,
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.edgecolor": "#d8d7d2", "grid.color": "#ececea", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 11, "figure.dpi": 130,
})

## Loading

One row per (size, depth, budget), stitching any `--seed-start` chunks back
together and deduplicating seeds. `cutoff_ok` is present only on cells that
`--validate` has processed; where it is missing the row is still shown, but
`n_unvalidated` records that its successes have not been checked against a
doubled Fock cutoff.

In [ ]:
def load_cells(sizes=SIZES):
    """Merge every multistart cell into one row per (size, depth, iters)."""
    rows = {}
    for size in sizes:
        for f in sorted(DATA.glob(f"multi_n{size}_*.npz")):
            d = np.load(f, allow_pickle=False)
            key = (size, int(d["depth"]), int(d["n_iters"]))
            r = rows.setdefault(key, {"size": size, "depth": key[1],
                                      "iters": key[2], "seed": [], "p": [],
                                      "energy": [], "conv": [], "ok": [],
                                      "hist": []})
            r["seed"].extend(np.asarray(d["seed"]).tolist())
            r["p"].extend(np.asarray(d["p_optimal"]).tolist())
            r["energy"].extend(np.asarray(d["energy"]).tolist())
            r["conv"].extend(np.asarray(d["conv_0p01"]).tolist())
            r["hist"].extend(np.asarray(d["history"]).tolist())
            # np.nan marks "not yet validated", distinct from False = "failed"
            ok = (np.asarray(d["cutoff_ok"], dtype=float) if "cutoff_ok" in d
                  else np.full(np.asarray(d["p_optimal"]).size, np.nan))
            r["ok"].extend(ok.tolist())

    out = []
    for (size, depth, iters), r in rows.items():
        _, keep = np.unique(np.asarray(r["seed"]), return_index=True)
        p = np.asarray(r["p"])[keep]
        ok = np.asarray(r["ok"], dtype=float)[keep]
        solved = p >= SUCCESS
        out.append({
            "size": size, "depth": depth, "iters": iters, "n": p.size,
            "p": p, "energy": np.asarray(r["energy"])[keep],
            "hist": np.asarray(r["hist"])[keep],
            "solved": int(solved.sum()), "rate": float(solved.mean()),
            "budget_used": float(np.mean(np.asarray(r["conv"])[keep]) / iters),
            "n_failed_cutoff": int(np.sum(solved & (ok == 0))),
            "n_unvalidated": int(np.sum(solved & np.isnan(ok))),
        })
    return sorted(out, key=lambda r: (r["size"], r["depth"], r["iters"]))


CELLS = load_cells()
print(f"{len(CELLS)} cells, sizes {sorted({c['size'] for c in CELLS})}")
for c in CELLS:
    warn = ""
    if c["n_failed_cutoff"]:
        warn = f"  <-- {c['n_failed_cutoff']} solved seeds FAIL the cutoff check"
    elif c["n_unvalidated"]:
        warn = f"  ({c['n_unvalidated']} solved seeds not yet validated)"
    print(f"  n={c['size']:2d} d={c['depth']:2d} it={c['iters']:6d} "
          f"{c['solved']}/{c['n']} solved  budget_used={c['budget_used']:.2f}{warn}")

## Figure 1 — the iteration budget is adequate at n=8 and not at n=12

`budget_used` at the shared 8000-step budget, one bar per depth. Above ~0.9 the
median run was still descending when its budget ended, so the number reported
for that cell describes a truncated optimization rather than the ansatz.

In [ ]:
BASE_ITERS = 8000
fig, ax = plt.subplots(figsize=(7.2, 3.6))

groups = [(s, [c for c in CELLS if c["size"] == s and c["iters"] == BASE_ITERS])
          for s in SIZES]
groups = [(s, cs) for s, cs in groups if cs]
width, gap, x = 0.62, 1.1, 0.0
ticks, labels = [], []
for size, cs in groups:
    xs = [x + i * width for i in range(len(cs))]
    ax.bar(xs, [c["budget_used"] for c in cs], width * 0.82,
           color=SERIES[size], zorder=3)
    for xi, c in zip(xs, cs):
        # direct labels: identity is never carried by colour alone
        ax.text(xi, c["budget_used"] + 0.02, f"d{c['depth']}", ha="center",
                va="bottom", fontsize=8, color=INK2)
    ticks.append(np.mean(xs))
    labels.append(f"n = {size}")
    x = xs[-1] + width + gap

ax.axhline(0.9, color="#e34948", lw=1.4, ls=(0, (4, 3)), zorder=4)
# annotate on the left, where the n=8 bars leave headroom -- on the right it
# collides with the depth labels of the tallest group
ax.text(ax.get_xlim()[0] + 0.08, 0.915, "still descending at the final step",
        ha="left", va="bottom", fontsize=8.5, color="#e34948")
ax.set_xticks(ticks, labels)
ax.set_ylim(0, 1.12)
ax.set_ylabel("fraction of budget used before\nthe run stopped improving")
ax.set_title(f"A fixed {BASE_ITERS:,}-step budget stops being enough as the "
             "problem grows", loc="left", pad=12)
ax.grid(axis="y", zorder=0)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

## Figure 2 — success rate over depth and budget

Random starts reaching P(optimal) >= 0.9. Cell labels give solved/total, so the
sample size is visible rather than hidden behind a colour.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

CMAP = LinearSegmentedColormap.from_list("seq_blue", SEQ)
sizes_present = [s for s in SIZES if any(c["size"] == s for c in CELLS)]
fig, axes = plt.subplots(1, len(sizes_present),
                         figsize=(3.5 * len(sizes_present), 3.5), squeeze=False)

for ax, size in zip(axes[0], sizes_present):
    cs = [c for c in CELLS if c["size"] == size]
    depths = sorted({c["depth"] for c in cs})
    iters = sorted({c["iters"] for c in cs})
    grid = np.full((len(depths), len(iters)), np.nan)
    for c in cs:
        grid[depths.index(c["depth"]), iters.index(c["iters"])] = c["rate"]

    ax.imshow(grid, cmap=CMAP, vmin=0, vmax=1, origin="lower", aspect="auto")
    for c in cs:
        i, j = depths.index(c["depth"]), iters.index(c["iters"])
        # ink flips on the dark end of the ramp so the label stays legible
        ax.text(j, i, f"{c['solved']}/{c['n']}", ha="center", va="center",
                fontsize=9, color="#ffffff" if c["rate"] > 0.55 else INK)
    ax.set_xticks(range(len(iters)), [f"{i//1000}k" for i in iters])
    ax.set_yticks(range(len(depths)), [str(d) for d in depths])
    ax.set_xlabel("iterations")
    ax.set_title(f"n = {size}", loc="left", color=SERIES[size])
    ax.grid(False)
axes[0][0].set_ylabel("ECD depth")
fig.suptitle("Seeds reaching P(optimal) >= 0.9, out of those run",
             x=0.02, ha="left", fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()

## Figure 3 — what truncation looks like

Median energy against iteration at n=12, one curve per budget, shaded to the
interquartile range across seeds. The longer budget settles lower -- and the
gap between the two end points is the part of the descent the 8000-step runs
never reach. The y-axis excludes the first few percent of each run, where the
energy falls from ~2000 and would flatten everything else into one line. The x-axis is the fraction of each run's own
budget, so the two schedules are comparable -- the learning rate decays at
`0.5 * n_iters` and `0.82 * n_iters`, so a 32000-step run is not an 8000-step
run continued.

In [ ]:
TRACE_SIZE = 12
cands = [c for c in CELLS if c["size"] == TRACE_SIZE]
best_depth = max(cands, key=lambda c: (c["rate"], c["depth"]))["depth"] if cands else None
traces = sorted([c for c in cands if c["depth"] == best_depth],
                key=lambda c: c["iters"])

fig, ax = plt.subplots(figsize=(7.2, 3.8))
lo, hi = np.inf, -np.inf
for c, colour in zip(traces, [SERIES[8], SERIES[10], SERIES[12]]):
    h = np.asarray(c["hist"])
    frac = np.linspace(0, 1, h.shape[1])
    med = np.median(h, axis=0)
    q1, q3 = np.percentile(h, 25, axis=0), np.percentile(h, 75, axis=0)
    ax.fill_between(frac, q1, q3, color=colour, alpha=0.16, lw=0)
    ax.plot(frac, med, color=colour, lw=2,
            label=f"{c['iters']:,} iterations  ({c['solved']}/{c['n']} solved)")
    # The first ~2% of a run falls from E ~ 2000 to near zero. Keeping that in
    # frame flattens everything that matters into a single line at the bottom,
    # so the window is set from the second half of the descent instead.
    tail = h[:, h.shape[1] // 2:]
    lo, hi = min(lo, np.percentile(tail, 5)), max(hi, np.percentile(tail, 95))

pad = 0.12 * (hi - lo)
ax.set_ylim(lo - pad, hi + pad)
# End labels are staggered: at convergence the two curves sit almost on top of
# one another, which is the point of the figure but ruins an inline label.
for k, (c, colour) in enumerate(zip(traces, [SERIES[8], SERIES[10]])):
    med = np.median(np.asarray(c["hist"]), axis=0)
    ax.annotate(f"{c['iters']//1000}k", (1.0, med[-1]),
                textcoords="offset points", xytext=(8, 10 if k == 0 else -12),
                va="center", fontsize=9, color=colour, fontweight="medium")

ax.set_xlabel("fraction of that run's own iteration budget")
ax.set_ylabel("energy (median, IQR shaded)")
ax.set_title(f"n = {TRACE_SIZE}, depth {best_depth}: the longer budget settles "
             "at a lower energy", loc="left", pad=12)
ax.legend(frameon=False, loc="upper right", fontsize=9)
ax.grid(axis="y")
ax.set_axisbelow(True)
ax.set_xlim(0, 1.06)
fig.tight_layout()
plt.show()

## Figure 4 — both costs grow with problem size

Two panels rather than one with twin axes: these are different quantities on
different scales, and overlaying them would invent a crossing point that means
nothing. The second panel shows success rate rather than iteration count --
every size's best cell is at 32000 because that is the largest budget tested,
so charting it would describe the grid rather than the problem.

In [ ]:
# "Best" is the highest success rate, not the cheapest cell with a single
# success -- at 8 seeds one lucky start is 12%, which is not a configuration
# that solves anything. Ties break toward fewer gates, then fewer iterations.
best = []
for size in sizes_present:
    cs = [c for c in CELLS if c["size"] == size and c["solved"] > 0]
    if cs:
        best.append(max(cs, key=lambda c: (c["rate"], -c["depth"], -c["iters"])))

# The second panel is success rate, not iteration count: every size's best cell
# sits at 32000 simply because that is the largest budget tested, so plotting it
# would show the edge of the grid rather than a property of the problem.
fig, (a1, a2) = plt.subplots(1, 2, figsize=(7.6, 3.2))
xs = [b["size"] for b in best]
for ax, ys, lab, ttl, fmt in (
        (a1, [2 * b["depth"] for b in best], "ECD gates (2 x depth)",
         "Circuit cost grows", lambda y, b: f"{y}"),
        (a2, [100 * b["rate"] for b in best], "seeds solved (%)",
         "Reliability falls", lambda y, b: f"{b['solved']}/{b['n']}")):
    ax.plot(xs, ys, color=INK2, lw=1.2, zorder=2)
    for x, y, b in zip(xs, ys, best):
        ax.scatter([x], [y], s=64, color=SERIES[b["size"]], zorder=3)
        ax.annotate(fmt(y, b), (x, y), textcoords="offset points",
                    xytext=(0, 9), ha="center", fontsize=9, color=INK)
    ax.set_xticks(xs, [f"n = {x}" for x in xs])
    ax.set_ylabel(lab)
    ax.set_title(ttl, loc="left")
    ax.grid(axis="y")
    ax.set_axisbelow(True)
    ax.margins(y=0.22)
fig.suptitle("Best configuration found for each instance "
             "(all at 32,000 iterations)",
             x=0.02, ha="left", fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.92))
plt.show()

for b in best:
    print(f"n={b['size']:2d}: depth {b['depth']} ({2*b['depth']} ECD gates), "
          f"{b['iters']:,} iters, {b['solved']}/{b['n']} solved")

## Table view

Present because three light-mode series sit below 3:1 contrast on the light
surface, and because a reader should be able to check any number in the figures
against the value it came from.

In [ ]:
hdr = (f"{'n':>3} {'depth':>6} {'gates':>6} {'iters':>7} {'seeds':>6} "
       f"{'solved':>7} {'rate':>6} {'budget_used':>12}  status")
print(hdr)
print("-" * len(hdr))
for c in CELLS:
    if c["n_failed_cutoff"]:
        status = f"{c['n_failed_cutoff']} solved seeds fail cutoff check"
    elif c["n_unvalidated"]:
        status = "not yet validated"
    else:
        u = c["budget_used"]
        status = ("BUDGET-LIMITED" if u > 0.9 else
                  "budget adequate" if u < 0.75 else "budget marginal")
    print(f"{c['size']:>3} {c['depth']:>6} {2*c['depth']:>6} {c['iters']:>7} "
          f"{c['n']:>6} {c['solved']:>7} {c['rate']:>5.0%} "
          f"{c['budget_used']:>12.2f}  {status}")

## What to say about this

1. **The 8000-step budget was never checked, and it fails at n >= 12.**
   `budget_used` runs 0.57-0.70 at n=8, 0.74-0.91 at n=10, 0.91-0.97 at n=12.
   Every CV-DV number published at 12 and 16 variables was measured on runs
   that had not converged.

2. **The deterministic initializer picked the wrong depth.** At n=12 the
   production knee is depth 16, chosen by the golden-angle sweep. Under random
   starts -- how the ansatz is actually deployed -- depth 16 gives 0/8 even at
   32000 iterations, while depth 20 gives 2/8. So the archived 2/50 = 4% was
   measured at a depth random starts cannot use *and* on a truncated budget.

3. **Fixing both raises n=12 from ~4% to ~25%,** a roughly 6x improvement,
   subject to the seed-count caveat. This can only widen the CV-DV vs DV gap,
   because DV converges by ~2000 steps and is unaffected by the budget.

4. **The cutoff is not implicated.** Every validated seed reproduces its
   P(optimal) at twice the Fock cutoff, and `tail_population` is ~1e-34.

### Open

- 50-seed runs at each winning cell (queued) -- 8 seeds cannot separate 50%
  from 62%.
- n=12 at depth 24 / 55000, since `budget_used` is still 0.87 at the current best.
- The DV arm at a matched budget, which is what makes the comparison publishable.
- n=16, parked: never solved at any depth or budget tried.